In [ ]:
# Use the list of cities to create site inputs for openeo
# Download sentinel-2 assets based on time, cloudcover and image quality
# Add band math operations
# https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel/sentinel-2/

# Updated helper files: prepare_openeo.py, utlilities.py
# RTS, May 16 - 20, 2025
# Reused July 9, 2025 to test open mine sites


In [1]:
# variables specific to your CoLabsetup ----------------------------------------
from google.colab import drive
import os, sys
drive.mount('/content/drive')
root = '/content/drive/MyDrive/'
sys.path.append(root +"Colab/research/code/")
datapath = root + "Colab/research/data/"
datapathnudge = root + "Projects/Nudge-X/geo_images/samples/"
datapathcities = root + "Projects/Nudge-X/sites/"
datapathmines = root + "Projects/Nudge-X/sites/"

Mounted at /content/drive


In [2]:
# install packages
%%capture
!pip install openeo --upgrade
!pip install gdal
!pip install rasterio --upgrade
!pip install scikit-image --upgrade

In [3]:
# imports
import openeo, json
import rasterio, numpy
from rasterio.plot import show

# home brew
from prepare_openeo import *
from qualcheck_module import *
from utilities import *

In [4]:
# City names and GPS coordinages
#collection = "Capital_Cities_with_Countries_and_Coordinates_update5.csv"
collection = "Open_Mines_Coordinates_v2.csv"
file_path = datapathmines + collection

In [5]:
category = "mines" #"cities
cities, bboxes = get_sites(file_path, category, limit=15)
print(cities)
print(bboxes)

['RedLakeMines', 'CrownestPass', 'KingsMountainLithiumMine', 'BeauvoirQuarry', 'KatangaCopper', 'MutshatshaMine', 'FimistonOpenPit', 'TancoMine', 'FooteLithiumMine', 'KokotokayAltayMIne', 'BurnagaMine', 'AidyrlaGoldDeposit', 'BishaMine', 'ImourarenUraniumMine', 'HoucaoBauxiteDeposit']
[[-93.8212626441203, 51.10902708029594, -93.67816135587971, 51.01909491970407], [-114.7451608789667, 49.679805080295935, -114.60630312103329, 49.589872919704064], [-81.40893372473143, 35.267746080295936, -81.29884627526859, 35.177813919704064], [2.8889284440116643, 46.22390608029593, 3.018811555988336, 46.13397391970406], [25.186550594777994, -10.972447919704065, 25.278171405222007, -11.062380080295936], [25.34821328302438, -10.668064919704065, 25.43974071697562, -10.757997080295937], [121.45312281064093, -30.732505919704064, 121.55779718935906, -30.822438080295935], [-95.51655820113047, 50.477029080295935, -95.37537579886953, 50.387096919704064], [-81.4013810733653, 35.26826008029594, -81.29129292663471,

In [6]:
# Connect to the cdse backend
# marcbohlen@gmail	!2EU ... EU2!
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")

#authenticate with your Copernicus credentials
connection.authenticate_oidc()
print()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=WZPD-PIKM 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.



In [84]:
# Make the start time variable depending on whether the first attempt returned a viable result

satellite = "SENTINEL2_L1C"
max_cloud = 5
start = "2024-10-01"
end = "2024-12-31"
band_selection = ["B12", "B08", "B04", "B03", "B02"]

#OK
#aspect = "rgb"
#aspect = "ndvi"
#aspect = "nbr"
#aspect = "ndbi"
#aspect = "fmi"

aspect = "rgb"
job_title = "nudge_" + aspect

In [85]:
## ------------------------------------------------------------------------------------------------------------------------------------------
# get just one of the items and create a job for OpenEO
## ------------------------------------------------------------------------------------------------------------------------------------------
# create_job_type is a function in "prepare_openeo.py" (in the code directory)
index = 14
bbox_s = bboxes[index]
city_s = cities[index]
job = create_job_type(connection, bbox_s, start, end, satellite, band_selection, max_cloud, job_title, aspect)

print(job_title)
print("SITE: ", city_s)
print("TIMEFRAME: ", start, end)
print(connection.list_jobs())

nudge_rgb
SITE:  HoucaoBauxiteDeposit
TIMEFRAME:  2024-10-01 2024-12-31
[{'created': '2025-07-16T21:23:47Z', 'id': 'j-2507162123474eb989abb8a8f8d4e55f', 'progress': 100, 'status': 'finished', 'title': 'nudge_rgb', 'updated': '2025-07-16T21:25:08Z'}, {'created': '2025-07-16T21:18:21Z', 'id': 'j-25071621182147f48ab83012334ad7f7', 'progress': 100, 'status': 'finished', 'title': 'nudge_rgb', 'updated': '2025-07-16T21:22:09Z'}, {'created': '2025-07-16T21:12:23Z', 'id': 'j-250716211223492fa02fd6049f328d25', 'progress': 100, 'status': 'finished', 'title': 'nudge_rgb', 'updated': '2025-07-16T21:16:13Z'}, {'created': '2025-07-16T21:07:28Z', 'id': 'j-250716210728417db14ee7fe41a501b6', 'progress': 100, 'status': 'finished', 'title': 'nudge_rgb', 'updated': '2025-07-16T21:09:14Z'}, {'created': '2025-07-16T21:07:13Z', 'id': 'j-250716210713484492ec2a3613c9b72c', 'progress': 0, 'status': 'created', 'title': 'nudge_rgb', 'updated': '2025-07-16T21:07:13Z'}, {'created': '2025-07-16T21:01:47Z', 'id': 'j-

In [86]:
try:
    job.start_and_wait()
except:
    print("something went wrong")

0:00:00 Job 'j-2507162126174bd785d38ce45f12a2be': send 'start'
0:00:13 Job 'j-2507162126174bd785d38ce45f12a2be': created (progress 0%)
0:00:18 Job 'j-2507162126174bd785d38ce45f12a2be': created (progress 0%)
0:00:25 Job 'j-2507162126174bd785d38ce45f12a2be': created (progress 0%)
0:00:33 Job 'j-2507162126174bd785d38ce45f12a2be': created (progress 0%)
0:00:43 Job 'j-2507162126174bd785d38ce45f12a2be': created (progress 0%)
0:00:55 Job 'j-2507162126174bd785d38ce45f12a2be': running (progress N/A)
0:01:11 Job 'j-2507162126174bd785d38ce45f12a2be': running (progress N/A)
0:01:31 Job 'j-2507162126174bd785d38ce45f12a2be': running (progress N/A)
0:01:55 Job 'j-2507162126174bd785d38ce45f12a2be': finished (progress 100%)


In [87]:
# Tally results and filter them. Check prepare_openeo.py
results = job.get_results()
metadata = results.get_metadata()
metadata['assets'].keys()

compiled = recompile(metadata)
res = filter(compiled)
print(compiled)
print(res)

{'openEO_2024-10-29Z.tif': {'B04': 100, 'B03': 100, 'B02': 100}, 'openEO_2024-11-03Z.tif': {'B04': 64.08, 'B03': 64.08, 'B02': 64.08}}
['openEO_2024-10-29Z.tif']


In [88]:
# Check the filtered list against the assets -
relevant_results = []

assets = results.get_assets()
for item in assets:
    for filtered in res:
        if(item.name == filtered):
            relevant_results.append(item)

# check
print(relevant_results)

#prune to get only the latest one...
latest_result = relevant_results[-1]

[<ResultAsset 'openEO_2024-10-29Z.tif' (type image/tiff; application=geotiff) at 'https://openeo.dataspace.copernicus.eu/openeo/1.2/jobs/j-2507162126174bd785d38ce45f12a2be/results/assets/ZTJjNjI3YjktMWQxNS00NDZlLWE4NDMtOWU3ZWIzYjExMDhh/b190517349842bc467947caf221e04ca/openEO_2024-10-29Z.tif?expires=1753306167'>]


In [89]:
#download those files and name them based on image type (aspect in job_title), location and date
targets = []

#for item in relevant_results:
item = latest_result
date = item.name.split("openEO_")[1]
date = date.split("Z.tif")[0] + ".tif"
if("rgb" in job_title):
    target = city_s + "_rgb_" + date
elif("ndvi" in job_title):
    target = city_s + "_ndvi_" + date
elif("nbr" in job_title):
    target = city_s + "_nbr_" + date
elif("fmi" in job_title):
    target = city_s + "_fmi_" + date
elif("ndbi" in job_title):
    target = city_s + "_ndbi_" + date
else:
    target = city_s + "_" + date

targets.append(target)
item.download(datapathcities + target)

PosixPath('/content/drive/MyDrive/Projects/Nudge-X/sites/HoucaoBauxiteDeposit_rgb_2024-10-29.tif')

In [90]:
print(targets)

['HoucaoBauxiteDeposit_rgb_2024-10-29.tif']


In [61]:
# Check the quality of the downloaded assets first...Only for the RGB images... if it is ok, the others should be as well...
for thing in targets:
    stats = evaluate_image_quality (datapathcities + thing)
    print(stats)

{'contrast': np.float64(5.6593300494493874), 'entropy': np.float64(71.99214812072084), 'sharpness': np.float64(12.026844512476275), 'noise': np.float64(90.61628579607711), 'overall': np.float64(45.0736521196809)}


In [91]:
# Pick one target to visualize here
# Make a loop to process them all...

pick = 0
source = targets[pick]
file_path = datapathcities + source
png_output_path = datapathcities + source.split(".tif")[0] + ".png"

# TEST
#test = "Islamabad_fmi_2025-01-24.tif"
#test = "Buenos Aires_rgb_2024-12-28.tif"
#file_path = datapathcities + test
#png_output_path = datapathcities + test.split(".tif")[0] + ".png"

print(file_path)

/content/drive/MyDrive/Projects/Nudge-X/sites/HoucaoBauxiteDeposit_rgb_2024-10-29.tif


In [92]:
# Adjust visualization for human consumption. Attention to stretch_contast - it can exaggerate results of band operations !
# Stretch_contrast on rgb images
brightness_factor = 1.2
saturation_factor = 0.65   #(0.75)
stretch_contrast = True
rgb = adjust_coloration(file_path, job_title, stretch_contrast, brightness_factor, saturation_factor)

In [93]:
# Display RGB / NDVI / NBR / NDBI /  FMI
import matplotlib.pyplot as plt

if rgb is not None:
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    #plt.imshow(rgb, vmin=-1, vmax=1)
    #plt.title("Color-Corrected GeoTIFF (RGB) of " +  target)
    plt.axis('off')
    plt.savefig(png_output_path, bbox_inches='tight', pad_inches=0)
    print(f"Saved color-corrected PNG to: {png_output_path}")
else:
    print(f"Skipping plotting for {source} as image data is None.")

Output hidden; open in https://colab.research.google.com to view.